# Dermatology: benefit of a second reader across all referral shares

Follows `data/derm/derm_referral_share_plan.json`. For referral shares 20% to 90%, compares the deployed model with the pair at the same share: all different-architecture pairs, and each model's strongest partner (chosen by internal validation AUC, no test labels). HIBA melanoma and any cancer; PAD any cancer. Patient-cluster bootstrap. CPU, about 10 minutes.

In [1]:
# Cell 1: load
from google.colab import drive
drive.mount('/content/drive')
import os, json, datetime, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
import numpy as np, pandas as pd
from scipy.stats import rankdata
DERM = '/content/drive/MyDrive/stochastic_orthogonality/data/derm'
FIG = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2/05_figures/synthesis'; os.makedirs(FIG, exist_ok=True)
CLASSES8 = ['MEL', 'BCC', 'SCC', 'AK', 'NV', 'BKL', 'DF', 'VASC']
ARCHS = ['resnet50', 'densenet121', 'efficientnet_b3', 'convnext_t', 'vit_b', 'vit_b_dinov2']; SOURCES = ['HAM10000', 'BCN20000']
MODELS = [f'{s}_{a}_seed{k}' for s in SOURCES for a in ARCHS for k in (0, 1)]; ARCH = {m: next(a for a in ARCHS if f'_{a}_seed' in m) for m in MODELS}
VAL = {m: v['val_malignant_auc'] for m, v in json.load(open(f'{DERM}/training_record.json')).items()}
PARTNERS = {a: [b for b in MODELS if ARCH[b] != ARCH[a]] for a in MODELS}
STRONGEST = {a: max(PARTNERS[a], key=lambda b: VAL[b]) for a in MODELS}
SETS = {}
for es in ('HIBA', 'PAD'):
    D = pd.read_csv(f'{DERM}/manifest_{es}.csv', dtype={'image_id': str, 'patient_id': str}).sort_values('image_id')
    D = D[~D.duplicated('raw_sha256', keep='first') & D.dx.isin(CLASSES8)].reset_index(drop=True)
    P = {m: pd.read_csv(f'{DERM}/predictions/{es}__{m}.csv', dtype={'image_id': str}).set_index('image_id').p_malignant.reindex(D.image_id).to_numpy() for m in MODELS}
    mal = D.dx.isin(['MEL', 'BCC', 'SCC']).to_numpy()
    SETS[es] = {'P': P, 'RK': {m: rankdata(P[m]) / len(D) for m in MODELS}, 'cl': pd.factorize(D.patient_id.astype(str))[0], 'f_prev': 1.31 * mal.mean(),
                'targets': {'any cancer': mal, **({'melanoma': D.dx.eq('MEL').to_numpy()} if es == 'HIBA' else {})}}
    print(es, len(D), 'images; prevalence-based share', round(SETS[es]['f_prev'], 3))
print('strongest partner by internal validation AUC, e.g.:', list(STRONGEST.items())[:3])

Mounted at /content/drive
HIBA 1616 images; prevalence-based share 0.609
PAD 2283 images; prevalence-based share 0.62
strongest partner by internal validation AUC, e.g.: [('HAM10000_resnet50_seed0', 'HAM10000_densenet121_seed0'), ('HAM10000_resnet50_seed1', 'HAM10000_densenet121_seed0'), ('HAM10000_densenet121_seed0', 'HAM10000_convnext_t_seed1')]


In [2]:
# Cell 2: sweep referral shares; pooled relative reduction with patient-cluster bootstrap
def refer(s, f):
    r = np.zeros(len(s), bool); r[np.argsort(-s, kind='stable')[:int(round(f * len(s)))]] = True; return r
GRID = [round(x, 2) for x in np.arange(0.20, 0.901, 0.05)]; RNG = np.random.default_rng(20261604); NB = 1000
ROWS = []
for es, S in SETS.items():
    ncl = S['cl'].max() + 1; W = [np.bincount(RNG.integers(0, ncl, ncl), minlength=ncl)[S['cl']].astype(float) for _ in range(NB)]
    for f in GRID + [round(S['f_prev'], 4)]:
        R = {m: refer(S['P'][m], f) for m in MODELS}; E = {}
        for a in MODELS:
            for b in PARTNERS[a]: E[(a, b)] = refer(np.maximum(S['RK'][a], S['RK'][b]), f)
        for tname, y in S['targets'].items():
            for label, pairs in (('all pairs', [(a, b) for a in MODELS for b in PARTNERS[a]]), ('strongest partner', [(a, STRONGEST[a]) for a in MODELS])):
                anc = sum((y & ~R[a]).astype(float) for a, b in pairs); ens = sum((y & ~E[(a, b)]).astype(float) for a, b in pairs)
                red = lambda w: (anc @ w - ens @ w) / (anc @ w) if anc @ w > 0 else np.nan
                bs = [red(w) for w in W]
                ROWS.append({'set': es, 'outcome': tname, 'pairs': label, 'share': f, 'prevalence_based': f == round(S['f_prev'], 4),
                             'relative_reduction': round(float(red(np.ones(len(y)))), 4), 'lo': round(float(np.nanpercentile(bs, 2.5)), 4),
                             'hi': round(float(np.nanpercentile(bs, 97.5)), 4), 'deployed_misses_per_model': round(float(anc.sum() / len(pairs)), 2)})
    print(es, 'done')
T = pd.DataFrame(ROWS); T['excludes_zero'] = T.lo > 0
print(T[~T.prevalence_based].groupby(['set', 'outcome', 'pairs']).agg(shares=('share', 'size'), shares_benefit_ci_above_0=('excludes_zero', 'sum'),
      min_reduction=('relative_reduction', 'min'), max_reduction=('relative_reduction', 'max')).to_string())

HIBA done
PAD done
                                   shares  shares_benefit_ci_above_0  min_reduction  max_reduction
set  outcome    pairs                                                                             
HIBA any cancer all pairs              15                         14         0.0086         0.5693
                strongest partner      15                         14         0.0179         0.5965
     melanoma   all pairs              15                         15         0.0434         0.4350
                strongest partner      15                         12         0.0150         0.5000
PAD  any cancer all pairs              15                         14         0.0063         0.2763
                strongest partner      15                         15         0.0256         0.3366


In [3]:
# Cell 3: figure and results
import matplotlib as mpl; mpl.use('Agg'); import matplotlib.pyplot as plt
mpl.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})
panels = [('HIBA', 'melanoma'), ('HIBA', 'any cancer'), ('PAD', 'any cancer')]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
for ax, (es, oc) in zip(axes, panels):
    for label, col in (('all pairs', '#0072B2'), ('strongest partner', '#D55E00')):
        s = T[(T.set == es) & (T.outcome == oc) & (T.pairs == label) & ~T.prevalence_based].sort_values('share')
        ax.plot(100 * s.share, 100 * s.relative_reduction, '-o', ms=3, color=col, label=label)
        ax.fill_between(100 * s.share, 100 * s.lo, 100 * s.hi, color=col, alpha=0.15)
    ax.axhline(0, color='#333', lw=0.8); ax.axvline(100 * SETS[es]['f_prev'], color='#555', ls='--', lw=1)
    ax.set_title(f'{es}: {oc}  (dashed: prevalence-based share)', loc='left', fontsize=9.5, weight='bold')
    ax.set_xlabel('Referral share (%)'); ax.set_ylabel('Reduction in missed cases vs deployed model (%)')
axes[0].legend(fontsize=8, frameon=False)
fig.suptitle('Skin cancer: benefit of a second reader at every referral share (same number of referrals as one model)', x=0.01, ha='left', fontsize=11, weight='bold')
fig.tight_layout(); fig.savefig(f'{FIG}/SuppFig_derm_referral_shares.png', dpi=300, bbox_inches='tight'); fig.savefig(f'{FIG}/SuppFig_derm_referral_shares.pdf', bbox_inches='tight'); plt.close(fig)
T.to_csv(f'{DERM}/derm_referral_share_results.csv', index=False)
summ = T[~T.prevalence_based].groupby(['set', 'outcome', 'pairs']).agg(shares=('share', 'size'), shares_with_ci_above_zero=('excludes_zero', 'sum'),
        min_reduction=('relative_reduction', 'min'), max_reduction=('relative_reduction', 'max')).reset_index()
json.dump({'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'plan': 'derm_referral_share_plan.json', 'status': 'exploratory robustness',
           'strongest_partner_rule': 'highest internal validation AUC among different-architecture models', 'summary': summ.to_dict(orient='records')},
          open(f'{DERM}/derm_referral_share_results.json', 'w'), indent=2)
print(summ.to_string()); print('Wrote derm_referral_share_results.json/.csv and 05_figures/synthesis/SuppFig_derm_referral_shares')

    set     outcome              pairs  shares  shares_with_ci_above_zero  min_reduction  max_reduction
0  HIBA  any cancer          all pairs      15                         14         0.0086         0.5693
1  HIBA  any cancer  strongest partner      15                         14         0.0179         0.5965
2  HIBA    melanoma          all pairs      15                         15         0.0434         0.4350
3  HIBA    melanoma  strongest partner      15                         12         0.0150         0.5000
4   PAD  any cancer          all pairs      15                         14         0.0063         0.2763
5   PAD  any cancer  strongest partner      15                         15         0.0256         0.3366
Wrote derm_referral_share_results.json/.csv and 05_figures/synthesis/SuppFig_derm_referral_shares
